# Laboratorio L8 - Il classificatore k-NN

B.6 - Data science e Machine Learning: dai dati ai modelli

File necessario: `pinguini.csv`.

Dati: Palmer Penguins, Horst AM, Hill AP, Gorman KB (2020), licenza CC0, https://allisonhorst.github.io/palmerpenguins/

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

def controlla(condizione, suggerimento="Non ancora: rileggere la consegna."):
    """Stampa un riscontro immediato sull'esercizio."""
    print("Corretto." if condizione else suggerimento)

pinguini = pd.read_csv("pinguini.csv").dropna(subset=["becco_lunghezza_mm", "pinna_lunghezza_mm", "massa_g"])
SPECIE = ["Adelie", "Chinstrap", "Gentoo"]
COLORI = {"Adelie": "#2a78d6", "Chinstrap": "#eb6834", "Gentoo": "#1baf7a"}
CHIARI = ["#cfe0f5", "#fad6c6", "#c4ecdd"]
MARCATORI = {"Adelie": "o", "Chinstrap": "s", "Gentoo": "^"}
print(len(pinguini), "pinguini con tutte le misure")

## 1. k-NN a mano (esercizio 1)

Sei pinguini di specie nota e un pinguino nuovo con becco 45,0 mm e pinna 195 mm.

Distanza euclidea tra due punti (x1, y1) e (x2, y2): radice di (x1 - x2)² + (y1 - y2)², cioè il teorema di Pitagora.

- `np.sqrt`: radice quadrata; `**2`: elevamento al quadrato
- operazioni su colonne intere: `(noti["becco"] - 45.0)` calcola la differenza per tutte le righe insieme

In [ ]:
noti = pd.DataFrame({
    "nome":   ["A1", "A2", "C1", "C2", "G1", "G2"],
    "becco":  [39.1, 41.1, 46.5, 49.5, 46.1, 50.0],
    "pinna":  [181, 198, 192, 200, 211, 220],
    "specie": ["Adelie", "Adelie", "Chinstrap", "Chinstrap", "Gentoo", "Gentoo"]})
nuovo_becco, nuova_pinna = 45.0, 195

noti["distanza"] = np.sqrt((noti["becco"] - nuovo_becco) ** 2 + (___) ** 2)
noti = noti.sort_values("distanza")
print(noti.round(2))
controlla(noti.iloc[0]["nome"] == "C1", "Manca la differenza tra le pinne: noti[\"pinna\"] - nuova_pinna.")

**Domande**: quale specie assegna il k-NN con k = 1? con k = 3? con k = 5? Che cosa succede con k = 5?

Risposte:

In [ ]:
for k in [1, 3, 5]:
    vicini = noti.head(k)
    print(f"k = {k}: vicini {list(vicini['nome'])} -> voti {vicini['specie'].value_counts().to_dict()}")

## 2. k-NN con scikit-learn (esercizio 2)

scikit-learn usa lo stesso schema per tutti i modelli:

1. si crea il modello scegliendo i suoi parametri: `KNeighborsClassifier(n_neighbors=5)`
2. si addestra con `fit(X, y)`: `X` è la tabella delle caratteristiche, `y` le etichette
3. si prevede con `predict(nuovi_X)`

Per il k-NN "addestrare" significa solo memorizzare gli esempi: il calcolo delle distanze avviene al momento della previsione.

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

X_noti = noti[["becco", "pinna"]]
modello = KNeighborsClassifier(n_neighbors=3)
modello.fit(X_noti, noti["specie"])
nuovo = pd.DataFrame({"becco": [45.0], "pinna": [195]})
print(modello.predict(nuovo))

Ora sull'intero dataset. Accuratezza: `score(X, y)` confronta le previsioni con le etichette vere.

Attenzione: qui l'accuratezza viene calcolata sugli stessi pinguini usati per l'addestramento. Per k = 1 ogni pinguino è il vicino più vicino di sé stesso. Il modo corretto di valutare è l'argomento di L10.

In [ ]:
X = pinguini[["becco_lunghezza_mm", "pinna_lunghezza_mm"]]
y = pinguini["specie"]
for k in [1, 5, 25]:
    m = KNeighborsClassifier(n_neighbors=k).fit(X, y)
    print(f"k = {k:2}: accuratezza sui dati di addestramento {m.score(X, y):.3f}")

## 3. Il confine di decisione al variare di k (esercizio 3)

`disegna_confine` colora il piano con la specie prevista. Il modello viene addestrato con `X.values` (solo i numeri, senza i nomi delle colonne) perché la griglia del disegno è una tabella di numeri.

In [ ]:
from matplotlib.colors import ListedColormap

def disegna_confine(prevedi, X, y, titolo=""):
    """Colora ogni punto del piano con la specie prevista e disegna sopra i pinguini veri.
    prevedi: funzione che riceve una tabella di punti (due colonne) e restituisce le specie previste."""
    a, b = X.iloc[:, 0], X.iloc[:, 1]
    ga, gb = np.meshgrid(np.linspace(a.min() - 1, a.max() + 1, 250),
                         np.linspace(b.min() - 2, b.max() + 2, 250))
    griglia = np.c_[ga.ravel(), gb.ravel()]                     # 62500 punti del piano
    previsti = np.array(prevedi(griglia))
    indici = np.array([SPECIE.index(s) for s in previsti]).reshape(ga.shape)
    plt.contourf(ga, gb, indici, levels=[-0.5, 0.5, 1.5, 2.5], cmap=ListedColormap(CHIARI))
    for s in SPECIE:
        sel = (y == s).to_numpy()
        plt.scatter(a[sel], b[sel], color=COLORI[s], marker=MARCATORI[s], s=18,
                    edgecolor="white", linewidth=0.5, label=s)
    plt.xlabel(X.columns[0]); plt.ylabel(X.columns[1]); plt.title(titolo)
    plt.legend(loc="lower right")
    plt.show()

In [ ]:
for k in [1, 5, 25]:
    m = KNeighborsClassifier(n_neighbors=k).fit(X.values, y)
    disegna_confine(m.predict, X, y, f"k-NN, k = {k}")

**Domande**: come cambia il confine aumentando k? Per quale k il confine segue anche i singoli pinguini "fuori posto"? Quale confine sembra più ragionevole per pinguini nuovi?

Risposte:

## 4. La scala delle caratteristiche (esercizio 4)

Lunghezza del becco (circa 32-60 mm) e massa (circa 2700-6300 g) hanno scale molto diverse: nella distanza la differenza di massa pesa centinaia di volte di più.

- Standardizzazione: per ogni caratteristica si sottrae la media e si divide per la deviazione standard; tutte le caratteristiche hanno così media 0 e deviazione standard 1
- `StandardScaler().fit(X)` calcola medie e deviazioni standard; `transform(X)` applica la trasformazione

In [ ]:
from sklearn.preprocessing import StandardScaler

X2 = pinguini[["becco_lunghezza_mm", "massa_g"]]
m_grezzo = KNeighborsClassifier(n_neighbors=5).fit(X2.values, y)
print("senza standardizzazione:", round(m_grezzo.score(X2.values, y), 3))
disegna_confine(m_grezzo.predict, X2, y, "becco e massa, senza standardizzazione")

In [ ]:
scala = StandardScaler().fit(X2.values)
X2_std = scala.transform(X2.values)
print("medie dopo la trasformazione:", X2_std.mean(axis=0).round(2), " dev. std.:", X2_std.std(axis=0).round(2))

m_std = KNeighborsClassifier(n_neighbors=5).fit(___, y)
acc_std = m_std.score(X2_std, y)
print("con standardizzazione:", round(acc_std, 3))
controlla(acc_std > 0.9, "Addestrare il modello sui dati standardizzati X2_std.")
disegna_confine(lambda punti: m_std.predict(scala.transform(punti)), X2, y, "becco e massa, con standardizzazione")

**Domanda**: perché senza standardizzazione le zone sono quasi strisce orizzontali?

Risposta:

## 5. Approfondimento (esercizio 5)

Usare tutte e quattro le misure (becco lunghezza e profondità, pinna, massa), standardizzate. L'accuratezza migliora? Perché con quattro caratteristiche non si può più disegnare il confine?

In [ ]:
misure = ["becco_lunghezza_mm", "becco_profondita_mm", "pinna_lunghezza_mm", "massa_g"]
X4 = StandardScaler().fit_transform(pinguini[misure])
m4 = KNeighborsClassifier(n_neighbors=5).fit(X4, y)
print("quattro misure:", round(m4.score(X4, y), 3))